In [ ]:
from sklearn.model_selection import KFold, train_test_split
import numpy as np
import pandas as pd

print("\n" + "="*60)
print("5-FOLD CROSS-VALIDATION (PROPOSED MODEL ONLY)")
print("="*60)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []
indices = np.arange(len(X_all))

for fold, (train_val_cal_idx, test_idx) in enumerate(kf.split(indices)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    tr_val, cal_idx = train_test_split(train_val_cal_idx, test_size=0.15, random_state=SEED)
    tr_idx, val_idx = train_test_split(tr_val, test_size=0.20, random_state=SEED)
    
    for target in TARGETS:
        bundle = make_bundle(
            tag=f"cv_fold{fold}_{target}",
            train_idx=tr_idx,
            validation_idx=val_idx,
            calibration_idx=cal_idx,
            test_idx=test_idx,
            target=target,
            seed=SEED
        )
        
        result = evaluate_model(
            name=PROPOSED,
            bundle=bundle,
            seed=SEED,
            tag=f"cv_fold{fold}_{target}_{PROPOSED}_s{SEED}"
        )
        
        metrics = get_metrics(bundle["y_test"], result["prediction"])
        
        cv_results.append({
            'Fold': fold + 1,
            'Target': target,
            'RMSE': metrics['RMSE'],
            'MAE': metrics['MAE'],
            'R2': metrics['R2']
        })
        
        print(f"  {target} | RMSE={metrics['RMSE']:.4f}  MAE={metrics['MAE']:.4f}  R2={metrics['R2']:.4f}")

df_cv = pd.DataFrame(cv_results)

print("\n" + "="*60)
print("5-FOLD CV SUMMARY (Mean ± Std)")
print("="*60)

cv_summary = df_cv.groupby(['Target'])[['RMSE', 'MAE', 'R2']].agg(['mean', 'std'])
cv_summary.columns = [f"{col[0]}_{col[1]}" for col in cv_summary.columns]
cv_summary = cv_summary.reset_index()

display(cv_summary)

cv_summary.to_csv(out('Table_CV_5Fold_Summary.csv'), index=False)
df_cv.to_csv(out('Table_CV_5Fold_Raw.csv'), index=False)

print(f"\n✅ Saved: Table_CV_5Fold_Summary.csv & Table_CV_5Fold_Raw.csv")